# Teachable machine

Intussen heb je de teachable machine van Google al eens uitgeprobeerd. In deze notebook gaan we dieper in op de werking van deze teachable machine. Je zal leren hoe je een bestaand AI-model kan aanpassen zodat het werkt voor de data die jij hebt. Deze techniek wordt meestal **transfer learning** genoemd. 

## Transfer learning

Bij transfer learning starten we van een bestaand AI-model. Dit model passen we aan voor onze taak. Hier baseren we ons op het ImageNet model. Dit model werd getraind met meer dan een miljoen afbeeldingen van 1000 verschillende objecten. Het is dus al zeer goed in het detecteren van deze objecten. In deze notebook voeg je een laag toe aan ImageNet en train je die laag om papier en PMD te herkennen.

Voor we kunnen starten, importeren we de nodige bibliotheken. Zowel de tensorflow, keras als sklear biblitheken bevatten verschillende functies die het makkelijke maken om met AI-modellen te werken. Tensorflow en Keras richten zich specifiek op neurale netwerken. Sklearn is een meer algemene bibliotheek waarmee je ook andere soorten AI-modellen kan bouwen. Daarnaast laden we ook de numpy bibliotheek in. Deze maakt het gemakkelijker om met matrices te werken. 

Verder hebben we bij Dwengo ook een aantal functies geschreven die het makkelijker maken om je data in te lezen en weer te geven in de notebook. Deze functies laden we in via de **helpers** module. Wil je de code eens van dichtbij bekijken, dan kan je het bestand *scripts/helpers.py* openen.

In [ ]:
# Installeer opencv
!pip install opencv-python

In [ ]:
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet import preprocess_input, decode_predictions
import numpy as np

from scripts import helpers

## Een dataset opstellen

Voor we een model kunnen aanpassen, hebben we een dataset nodig. Deze dataset zal afbeeldingen van papier en PMD bevatten. In de bestandsverkenner aan de linkerkant zie je een mapje *dataset* daarin vind je twee submapjes: *papier* en *PMD*. Deze mapjes bevatten de afbeeldingen waarop we gaan trainen.

In de volgende codecellen laden we de afbeeldingen in elk mapje in in een lijst. We hebben al een aantal functies voorzien die het makkelijker maken om de gegevens te verwerken, deze zitten in de *helpers* bibliotheek. Hieronder roepen we een functie op die twee parameters krijgt. De eerste parameter is het mapje met afbeeldingen van PMD, de tweede parameter is het label voor de items in die map. De functie zal de afbeeldingen in de map inladen en deze omzetten naar afbeeldingen van 240x240 pixels. 

In [ ]:
# We laden alle afbeeldingen in de map 'dataset/pmd' in en geven ze de label 'PMD'
afbeeldingen_pmd, labels_pmd = helpers.laadt_bestanden_in_map_met_label("dataset/pmd", label="PMD")

Nu we de afbeeldingen van PMD hebben ingeladen, bekijken we hoe deze eruitzien. In de cel hieronder zie je de code om verschillende eigenschappen van onze dataset weer te geven.

In [ ]:
print(f"De dataset bevat {len(afbeeldingen_pmd)} afbeeldingen met label 'PMD'")
print(f"De labels zijn: {labels_pmd}")
print(f"De eerste afbeelding heeft een grootte van {afbeeldingen_pmd[0].shape}")
print("De eerste zes afbeeldingen zien er als volgt uit:")
helpers.toon_afbeeldingen(afbeeldingen_pmd, labels_pmd, max_afbeeldingen=6)

**Opdracht:** Zoek in bovenstaande uitvoer de grootte van de afbeeldingen. Deze bestaat uit drie getallen. Wat denk je dat de betekenis is van elk van deze drie getallen?

**Opdracht**: Vul onderstaande codecellen aan zodat je de afbeeldingen van Papier opslaat in een variabele. Vervang op de gepaste plaatsen de ___ door de correcte code.

In [ ]:
# We laden alle afbeeldingen in de map 'dataset/papier' in en geven ze de label 'Papier'
afbeeldingen_papier, labels_papier = helpers.laadt_bestanden_in_map_met_label("___", label="___")

In [ ]:
print(f"De dataset bevat {len(___)} afbeeldingen met label 'Papier'")
print(f"De labels zijn: {___}")
print(f"De eerste afbeelding heeft een grootte van {___}")
print("De eerste zes afbeeldingen zien er als volgt uit:")
helpers.toon_afbeeldingen(___, ___, max_afbeeldingen=6)

## De data klaarmaken voor het AI-systeem

Nu we onze afbeeldingen en labels ingeladen hebben in Python, kunnen we deze verwerken tot een formaat dat het AI-systeem nodig heeft. Daarvoor doorlopen we de volgende stappen.
1. We voegen onze afbeeldingen van PMD en papier samen tot één dataset.
2. We zetten de labels om van tekst naar getallen.
3. We splitsen deze dataset op in drie verzamelingen.
    * **De trainingsverzameling**: deze gebruiken we om ons AI-systeem te trainen.
    * **De validatieverzameling**: deze gebruiken we om de prestatie van het AI-systeem tijdens de ontwikkeling te testen. De afbeeldingen in deze verzameling overlappen niet met de trainingsverzameling. Deze verzameling is nodig om te zien of het AI-systeem kan generaliseren en dus niet gewoon de afbeeldingen in de trainingsverzameling vanbuiten geleerd heeft.
    * **De testverzameling**: deze gebruiken we om de prestatie van het AI-systeem na de ontwikkeling te valideren. De afbeeldingen in deze verzameling overlappen niet met die in de train- en testverzamelingen. 
4. We verrijken de trainingsverzameling door de bestaande afbeeldingen te roteren, te herschalen en te spiegelen. Dit noemen we *data augmentation*.
    

Wil je weten waarom het nodig is om een train-, validatie- en testverzameling te hebben? Doorloop dan de activiteit over het herkennen van emoties op [dwengo.org/waisda](dwengo.org/waisda).

### Stap 1: het samenvoegen van de afbeeldingen en labels

Met de onderstaande code voegen we alle afbeeldignen en labels samen. Het resultaat zijn twee numpy arrays, een met de afbeeldingen en een met de labels.

In [ ]:
afbeeldingen = np.vstack([np.array(afbeeldingen_pmd), np.array(afbeeldingen_papier)])
labels = np.concatenate([np.array(labels_pmd), np.array(labels_papier)])

Druk informatie over de arrays af.

In [ ]:
print(f"De dataset bevat {afbeeldingen.shape[0]} afbeeldingen")
print(f"Er zijn {len(labels)} labels")
print(f"De eerste afbeelding heeft een grootte van {afbeeldingen[0].shape}")

**Opdracht**: Controleer het formaat van de dataset. Komt deze overeen met de som van het aantal afbeeldingen van PMD en papier?

### Stap 2: De labels omzetten van tekst naar getallen.

Omdat computers sneller en efficiënter kunnen rekenen met getallen, zetten we onze labels om van tekst naar getallen. Hier gebruiken we **one-hot** encodering. We zullen elk label voorstellen door twee getallen. Het eerste paar getallen is het label voor *PMD*. Hier is het eerste getal een 1 en het tweede een 0. Het tweede paar getallen is het label voor *Papier*. Hier is het eerste getal een 0 en het tweede een 1. Op onderstaande afbeelding zie je visueel hoe de labels voor PMD en Papier eruitzien.

![](images/voorbeeld_one_hot.png)

In [ ]:
# Deze code zal onze labels one-hot encoderen.
labels_one_hot = helpers.one_hot_encode_labels(labels, ["PMD", "Papier"])

Nu we de nieuwe labels hebben, kunnen we 10 willekeurige afbeeldingen afdrukken met hun nieuwe label.

In [ ]:
# Genereer 10 willekeurige indices.
random_indices = np.random.randint(0, len(labels), 10)
# Toon deze 10 willekeurige afbeeldingen.
helpers.toon_afbeeldingen(afbeeldingen[random_indices], labels_one_hot[random_indices], max_afbeeldingen=10)

### Stap 3: de dataset opsplitsen in train-, test- en validatieverzameling.

Om onze dataset op te splitsen in deze drie verzamelingen gebruiken we de functie *train_test_split* uit de *sklearn* bibliotheek. We gebruiken deze twee keer, eerst om een validatieverzameling op te stellen, daarna om een test- en trainverzameling op te stellen.

#### De validatieverzameling opstellen

Onderstaande codecel zal willekeurig 20% van de dataset selecteren als validatieset. 

In [ ]:
overige_afbeeldingen, validatie_afbeeldingen, overige_labels, validatie_labels = train_test_split(afbeeldingen, labels_one_hot, test_size=0.2)

Bekijk het formaat van de validatieverzameling en de verzameling met overige afbeeldingen.

In [ ]:
print(f"De overige dataset bevat {overige_afbeeldingen.shape[0]} afbeeldingen")
print(f"De validatieverzameling bevat {validatie_afbeeldingen.shape[0]} afbeeldingen")

**Opdracht**: Vul onderstaande code aan zodat de *overige_afbeeldingen* en *overige_labels* worden opgesplitst in trainingsverzameling en testverzameling. 10% van de overige afbeeldingen moet gebruikt worden als testverzameling.

In [ ]:
# Vul deze code aan op de plaatsen waar ___ staat.
train_afbeeldingen, test_afbeeldingen, train_labels, test_labels = train_test_split(___, ___, test_size=___)

In [ ]:
print(f"De trainingsverzameling bevat {___} afbeeldingen")
print(f"De testverzameling bevat {___} afbeeldingen")

### Stap 4: de dataset verrijken (data augmentation).

Omdat we hier werken met een relatief kleine dataset, kan het een goed idee zijn om deze dataset artificieel uit te breiden. Dat kunnen we doen door afbeeldingen in onze dataset licht te vervormen. Op die manier kan het AI-model ook leren hoe variaties van objecten in de dataset eruitzien. Dit zou ervoor moeten zorgen dat het model beter kan generaliseren en dus beter zal zijn in het herkennen van objecten dat het nog niet gezien heeft.

Hiervoor gebruiken we de *ImageDataGenerator* functie uit keras bibliotheek. Deze kunnen we gebruiken om automatisch variaties op een bestaande afbeelding te genereren. Deze variaties worden bekomen door de volgende operaties willekeurig uit te voeren op een afbeelding:
* Draaien
* Horizontaal verschuiven
* Verticaal verschuiven
* In- en uitzoomen
* Spiegelen


In [ ]:
datagenerator = ImageDataGenerator(
    rotation_range=20,      # Draai de afbeelding met maximaal 20 graden
    width_shift_range=0.2,  # Verschuif de afbeelding horizontaal met maximaal 20%
    height_shift_range=0.2, # Verschuif de afbeelding verticaal met maximaal 20%
    zoom_range=0.15,        # Zoom in and uit met 15%
    horizontal_flip=True,   # Spiegel de afbeelding horizontaal
    fill_mode="nearest"     # Vul de lege pixels op met de dichtstbijzijnde pixel
)

Deze datagenerator kunnen we meerdere keren toepassen op elke afbeelding in onze trainingsverzameling. De code hieronder zal voor elke afbeelding in de dataset 5 variaties genereren.

In [ ]:
augmented_afbeeldingen = []
augmented_labels = []

n_augmented = 5  # Aantal variaties per afbeelding

for i in range(len(train_afbeeldingen)):
    image = train_afbeeldingen[i]
    label = train_labels[i]
    
    # Voeg de originele afbeelding toe aan de augmented afbeeldingen lijst
    augmented_afbeeldingen.append(image)
    augmented_labels.append(label)
    
    # Geef de afbeelding de juiste vorm.
    image = np.expand_dims(image, axis=0)
    
    # Genereer n_augmented variaties van de afbeelding.
    aug_iter = datagenerator.flow(image, batch_size=1)
    for _ in range(n_augmented):
        aug_image = next(aug_iter)[0].astype('uint8')
        augmented_afbeeldingen.append(aug_image)
        augmented_labels.append(label)  # Gebruik hetzelfde label voor de gegenereerde afbeeldingen.

# Zet om naar numpy arrays.
augmented_afbeeldingen = np.array(augmented_afbeeldingen)
augmented_labels = np.array(augmented_labels)

**Opdracht:** Hoeveel afbeeldingen zitten er in de verrijkte dataset? Schrijf hieronder de code om dat te weten te komen.

**Opdracht:** Schrijf in de volgende cel code om 10 willekeurige afbeeldingen uit de verrijkte dataset te tonen op het scherm. Baseer je hiervoor op de code die we eerder al gebruikten om willekeurige afbeeldingen te tonen.

**Opdracht:** Voer de code die je hebt geschreven in de vorige cel meerdere keren uit. Je krijgt telkens 10 willekeurige afbeeldingen te zien uit de verrijkte dataset. Zie je wat *data augmentation* doet met de afbeeldingen?

## ImageNet inladen

We laden eerst en vooral het bestaande AI-model in. Dit model passen we verder in de notebook aan.

In [ ]:
imagenet = keras.applications.MobileNetV2(
    weights="imagenet",
)

We kunnen de structuur van het ImageNet netwerk bekijken met het volgende commando.

In [ ]:
print(imagenet.summary())

Je ziet dat het model 3 504 872 *Trainable params* ofwel trainbare parameters heeft. Dit zijn de gewichten van het netwerk die aangepast werden tijdens het trainingsproces.

In de tabel die weergegeven wordt, zie je heel wat rijen. Elke rij komt overeen met een laag in het neurale netwerk. Met het volgende commando kan je tellen hoeveel lagen het netwerk heeft.

In [ ]:
print(len(imagenet.layers))

Het ImageNet model kan 1000 verschillende objecten identificeren. Onderstaande code drukt een lijst af van deze objecten. Je zal zien dat de labels in het Engels zijn.

In [ ]:
helpers.druk_imagenet_labels_af()

**Opdracht:** Bekijk de lijst met de labels van ImageNet. Welke objecten vallen onder PMD en welke onder papier?

We kunnen het ImageNet model gebruiken om een voorspelling te doen op een aantal van de afbeeldingen in onze trainingsverzameling. Onderstaande code zal dat doen voor de eerste 10 afbeeldingen in onze trainingsverzameling.

In [ ]:
voorspellingen = imagenet.predict(augmented_afbeeldingen[0:10])

# Zet de voorspellingen om naar labels.
voorspellingen_met_label = decode_predictions(voorspellingen, top=1)

# Druk de voorspellingen af.
for i, voorspelling in enumerate(voorspellingen_met_label):
    (id, label, score) = voorspelling[0]
    print(f"Het model zegt met {score:.4f} zekerheid dat afbeelding {i} een {label} is.")


Je ziet dat het model eigenlijk niet goed weet wat er op de afbeeldingen te zien is. Toch is het nuttig om het model te hergebruiken voor onze taak. Het model is immers al in staat om bepaalde eigenschappen (bv. lijnen en vormen) uit de afbeeldingen te halen. Dat is ook voor onze toepassing nuttig.

## ImageNet uitbreiden

Voor onze taak, het herkennen van PMD en papier, gaan we het ImageNet model uitbreiden. 

In [ ]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

Hieronder voegen we twee nieuwe lagen toe aan het ImageNet model. Een *Dense* laag met 1024 neuronen en een *Dense* laag met 2 neuronen. 

In [ ]:
# Sla de laatste laag van het model op.
x = imagenet.output

# Voeg een nieuwe vollig verbonden laag toe met 1024 neuronen en een relu activatiefunctie.
x = Dense(1024, activation='relu')(x)

# Voeg een nieuwe laag toe die de voorspellingen doet van onze one-hot encoded labels.
predictions = Dense(2, activation='softmax')(x)  

# Combineer het bestaande model met de nieuwe lagen.
uitgebreid_model = Model(inputs=imagenet.input, outputs=predictions)

**Opdracht:** Druk het aantal lagen van het nieuwe model af. Klopt het dat het nieuwe model twee lagen meer heeft dan het ImageNet model?

In [ ]:
# Druk het aantal lagen in het uitgebreide model af.
print(len(uitgebreid_model.layers))

### Trainen van de nieuwe lagen

We willen nu enkel de nieuwe lagen in het model trainen. Dat kunnen we doen door de lagen van het originele ImageNet model te "bevriezen". Dit doen we door de *trainable* eigenschap van de lagen op *False* te zetten.

In [ ]:
# Bevries de lagen van het ImageNet model.
for laag in imagenet.layers:
    laag.trainable = False

Nu kunnen we de nieuwe lagen in het aangepaste model trainen met de afbeeldingen in onze trainingsverzameling. Wanneer je de volgende cel uitvoert, zal je zien dat het netwerk begint te leren op basis van onze trainingsverzameling. Je ziet verschillende informatie. 
* In de hoeveelste *Epoch* we zitten. Dit geeft aan hoeveel keer we de volledige trainingsverzameling als voorbeeld hebben gegeven aan het netwerk.
* De *accuracy* wordt berekend door het aantal correcte voorspellingen te delen door het totaal aantal voorspellingen. Hoe hoger de accuracy, hoe beter de prestatie van het netwerk op de trainingsverzameling. Je zal zien dat je zowel de *accuracy* op de trainingsverzameling (accuracy) als die op de validatieverzameling (val_accuracy) kan zien.
* De *loss* geeft aan hoeveel de voorspellingen van het netwerk gemiddeld afwijken van de correcte waarde. Hoe hoger de loss hoe slechter de prestatie van het netwerk op de trainingsverzameling. 

In [ ]:
uitgebreid_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
uitgebreid_model.fit(augmented_afbeeldingen, augmented_labels, epochs=10, batch_size=32, validation_data=(validatie_afbeeldingen, validatie_labels))

**Opdracht:** Bekijk de uitvoer van bovenstaande cel. Heeft het model geleerd hoe het afbeeldingen van papier en PMD moet onderscheiden?

### Het AI-systeem testen

Nu we een model hebben dat PMD en papier kan detecteren, kunnen we het resultaat ervan uittesten op onze testverzameling.

In [ ]:
# Bereken de accuracy op de testverzameling.
test_loss, test_accuracy = uitgebreid_model.evaluate(test_afbeeldingen, test_labels)
print(f"Test accuracy: {test_accuracy}")

In [ ]:
# Doe een voorspelling voor de afbeeldingen in de testverzameling.
predictions = uitgebreid_model.predict(test_afbeeldingen)

In [ ]:
# Toon de afbeeldingen met hun voorspelling.
mapped_labels_true = ["PMD" if np.argmax(label) == 0 else "Papier" for label in test_labels]
mapped_labels_predicted = ["PMD" if np.argmax(label) == 0 else "Papier" for label in predictions]
mapped_labels_combined = [f"Echt: {mapped_labels_true[i]} \n Voorspeld: {mapped_labels_predicted[i]}" for i in range(len(mapped_labels_true))]
helpers.toon_afbeeldingen(test_afbeeldingen, mapped_labels_combined, max_afbeeldingen=len(test_afbeeldingen))

### Imagenet tunen (optioneel)

We hebben nu een model waarvan de laatste twee lagen getraind zijn om PMD en Papier te herkennen. Om deze twee lagen wat meer af te stemmen op de lagen in het ImageNet model, kunnen we het ImageNet model zelf ook bijtrainen. Dat doen we door te trainen met een lage *learning rate* zo passen we de gewichten in het bestaande ImageNet maar een klein beetje aan. Op die manier behoudt het model zijn kracht maar is het toch meer afgestemd op onze nieuwe taak.

Eerst "ontdooien" we de laatste 20 lagen van het origineel model, deze die we eerder "bevroren" hebben. Dat doen we door de *trainable* eigenschap ervan op *True* te zetten.

In [ ]:
for laag in imagenet.layers[-20:]:
    laag.trainable = True

Nu trainen we ons model verder maar met een kleine *learning rate*. Deze *learning rate* geeft aan hoeveel de gewichten in het model aangepast worden tijdens het trainen. Door deze waarde klein te kiezen, zal je geen grote veranderingen doen aan de gewichten. Je zal ze enkel wat beter op elkaar afstemmen.

In [ ]:
# Maak het model klaar om verder te trainen met een lagere learning rate.
uitgebreid_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# Train het model verder.
history_fine = uitgebreid_model.fit(augmented_afbeeldingen, augmented_labels, epochs=10, batch_size=32, validation_data=(validatie_afbeeldingen, validatie_labels))

## Het model opslaan

Nu we de gewichten van ons model hebben aangepast aan de hand van onze data, kunnen we het model opslaan in een bestand. Dat bestand kunnen we dan downloaden en in een andere applicatie terug importeren. Om het model op te slaan in een bestand gebruiken we onderstaande code.

In [ ]:
# Save model to file
uitgebreid_model.save("model.h5")

Je zal zien dat er een nieuw bestand is bijgekomen links in de bestandverkenner. Dat bestand kan je downloaden en gebruiken in je eigen Python applicatie. Wil je het model uitproberen op een live video stream van de webcam? Dan kan je daarvoor het bestand *test_het_model_lokaal.py* downloaden dat je in de *scripts* map kan vinden. Door dat script, samen met het opgeslagen model, in een map te zetten op je computer en het Python script uit te voeren, kan je voorspellingen doen op een live video stream van je webcam.

**Opdracht:** Download je model en het bestand *test_het_model_lokaal.py". Probeer het script op je eigen computer. Merk op dat je daarvoor de nodige bibliotheken zoals opencv, tensorflow, keras en numpy zal moeten installeren. Dat kan met de volgende commando's:

* pip install numpy
* pip install --upgrade keras
* python3 -m pip install tensorflow-cpu
* pip install opencv-python